# 03 -- TLS et certificats

## Objectifs pedagogiques

A la fin de ce notebook, vous saurez :

- comprendre TLS 1.2/1.3 et son handshake ;
- generer un certificat auto-signe avec `cryptography` ;
- creer une CSR (Certificate Signing Request) ;
- signer des certificats en tant qu'autorite (CA) ;
- inspecter un certificat PEM/DER ;
- configurer un client/serveur HTTPS en Python ;
- comprendre les chaines de certification ;
- gerer l'expiration et la rotation.

## Prerequis -- ce que vous devez maitriser

- les bases de la cryptographie (hash, cle pub/priv) ;
- HTTP et reseau ;
- le paquet `cryptography` (voir `01_Cryptography_complet.ipynb`).

Ce que ce notebook **ne couvrira pas** :

- Let's Encrypt / ACME en detail ;
- mTLS avance pour kubernetes ;
- TLS 1.0/1.1 (obsoletes).

## Plan

1. Rappels cryptographie
2. Le handshake TLS
3. Generer une cle privee
4. Certificat auto-signe
5. Inspecter un certificat
6. CSR : Certificate Signing Request
7. CA : signer des certificats
8. Chaine de certification
9. Serveur HTTPS Python
10. Client HTTPS avec verification
11. Fichiers PEM vs DER
12. Expiration et rotation
13. mTLS (intro)
14. Synthese
15. Exercices

---

## 1. Rappels cryptographie

- **Cle privee** : reste secrete, signe et dechiffre
- **Cle publique** : partagee, verifie et chiffre
- **Certificat X.509** : cle publique + identite + signature d'une autorite
- **CA (Certificate Authority)** : entite de confiance (Let's Encrypt, Digicert...)
- **Chain of trust** : navigateur fait confiance a la CA -> certif signe par la CA est accepte

---

## 2. Le handshake TLS

### TLS 1.3 handshake (simplifie)

```
Client                         Server
  |                                |
  |------ ClientHello ----------->|  (algos supportes)
  |                                |
  |<----- ServerHello ------------|
  |<----- Certificate ------------|  (certif serveur)
  |<----- Finished ---------------|
  |                                |
  |------ Finished -------------->|
  |                                |
  |<====== application data ====>|  (chiffre AES-GCM)
```

TLS 1.3 est **plus rapide** (1 RTT au lieu de 2) et **plus sur** (pas de suites faibles).

---

## 3. Generer une cle privee

In [ ]:
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.hazmat.primitives import serialization

key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
pem = key.private_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption(),
)
print(pem[:100].decode())

In [ ]:
# Avec EC (Elliptic Curve) - plus rapide et cle plus courte
from cryptography.hazmat.primitives.asymmetric import ec
eckey = ec.generate_private_key(ec.SECP384R1())
print('EC key SECP384R1 generee')

---

## 4. Certificat auto-signe

In [ ]:
from cryptography import x509
from cryptography.x509.oid import NameOID
from cryptography.hazmat.primitives import hashes
import datetime

subject = issuer = x509.Name([
    x509.NameAttribute(NameOID.COUNTRY_NAME, 'FR'),
    x509.NameAttribute(NameOID.ORGANIZATION_NAME, 'DemoCorp'),
    x509.NameAttribute(NameOID.COMMON_NAME, 'demo.local'),
])

now = datetime.datetime.now(datetime.UTC)
cert = (
    x509.CertificateBuilder()
    .subject_name(subject)
    .issuer_name(issuer)
    .public_key(key.public_key())
    .serial_number(x509.random_serial_number())
    .not_valid_before(now)
    .not_valid_after(now + datetime.timedelta(days=365))
    .add_extension(x509.SubjectAlternativeName([x509.DNSName('demo.local'), x509.DNSName('localhost')]), critical=False)
    .sign(key, hashes.SHA256())
)
cert_pem = cert.public_bytes(serialization.Encoding.PEM)
print(cert_pem[:120].decode())

---

## 5. Inspecter un certificat

In [ ]:
print('Subject:', cert.subject)
print('Issuer :', cert.issuer)
print('Valid from:', cert.not_valid_before_utc)
print('Valid to  :', cert.not_valid_after_utc)
print('Serial:', cert.serial_number)

In [ ]:
# Extensions
for ext in cert.extensions:
    print(ext.oid._name, '->', ext.value)

---

## 6. CSR : Certificate Signing Request

In [ ]:
csr = (
    x509.CertificateSigningRequestBuilder()
    .subject_name(x509.Name([
        x509.NameAttribute(NameOID.COMMON_NAME, 'client.example.com'),
    ]))
    .add_extension(x509.SubjectAlternativeName([x509.DNSName('client.example.com')]), critical=False)
    .sign(key, hashes.SHA256())
)
print(csr.public_bytes(serialization.Encoding.PEM)[:120].decode())

---

## 7. CA : signer des certificats

In [ ]:
# 1. Cle + certif CA (root)
ca_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
ca_name = x509.Name([x509.NameAttribute(NameOID.COMMON_NAME, 'DemoRootCA')])
ca_cert = (
    x509.CertificateBuilder()
    .subject_name(ca_name)
    .issuer_name(ca_name)
    .public_key(ca_key.public_key())
    .serial_number(x509.random_serial_number())
    .not_valid_before(now)
    .not_valid_after(now + datetime.timedelta(days=3650))
    .add_extension(x509.BasicConstraints(ca=True, path_length=None), critical=True)
    .sign(ca_key, hashes.SHA256())
)
print('CA creee')

In [ ]:
# 2. Signer le CSR avec la CA
signed = (
    x509.CertificateBuilder()
    .subject_name(csr.subject)
    .issuer_name(ca_cert.subject)
    .public_key(csr.public_key())
    .serial_number(x509.random_serial_number())
    .not_valid_before(now)
    .not_valid_after(now + datetime.timedelta(days=90))
    .sign(ca_key, hashes.SHA256())
)
print('Certif signe par CA')

---

## 8. Chaine de certification

Un navigateur valide : `certif serveur` -> `certif intermediaire` -> `certif root`. Le root est dans le **trust store** du systeme/navigateur.

In [ ]:
# Concatener en PEM pour construire une chaine
chain = signed.public_bytes(serialization.Encoding.PEM) + ca_cert.public_bytes(serialization.Encoding.PEM)
print(len(chain), 'octets de chaine PEM')

---

## 9. Serveur HTTPS Python

In [ ]:
# Sauvegarder cert + key sur disque
import tempfile, pathlib
tmp = pathlib.Path(tempfile.gettempdir())
(tmp / 'demo.crt').write_bytes(cert.public_bytes(serialization.Encoding.PEM))
(tmp / 'demo.key').write_bytes(pem)
print('Ecrit dans', tmp)

In [ ]:
# Serveur HTTPS minimal avec http.server
import ssl
import http.server

# ctx = ssl.create_default_context(ssl.Purpose.CLIENT_AUTH)
# ctx.load_cert_chain(certfile=str(tmp / 'demo.crt'), keyfile=str(tmp / 'demo.key'))
# server = http.server.HTTPServer(('localhost', 4443), http.server.SimpleHTTPRequestHandler)
# server.socket = ctx.wrap_socket(server.socket, server_side=True)
# server.serve_forever()
print('Serveur HTTPS defini (ne pas lancer en notebook)')

---

## 10. Client HTTPS avec verification

In [ ]:
import ssl
ctx = ssl.create_default_context()
ctx.minimum_version = ssl.TLSVersion.TLSv1_2
print('min TLS :', ctx.minimum_version)

In [ ]:
# Ajouter une CA de confiance
# ctx.load_verify_locations(cadata=ca_cert.public_bytes(serialization.Encoding.PEM).decode())
print('On peut ajouter des CA custom')

In [ ]:
import httpx
# httpx.get('https://example.com', verify=ctx)
print('httpx supporte verify=ctx')

---

## 11. Fichiers PEM vs DER

| Format | Description |
|---|---|
| **PEM** | Texte base64, balises `-----BEGIN CERTIFICATE-----`, le plus courant |
| **DER** | Binaire pur, plus compact |
| **PKCS#12 (.pfx)** | Conteneur chiffre contenant cles et certifs |

Conversions avec `openssl` :

```bash
openssl x509 -in cert.pem -outform DER -out cert.der
openssl pkcs12 -in bundle.pfx -out bundle.pem -nodes
```

In [ ]:
# En Python
der = cert.public_bytes(serialization.Encoding.DER)
print('DER :', len(der), 'octets, PEM :', len(cert.public_bytes(serialization.Encoding.PEM)), 'octets')

---

## 12. Expiration et rotation

In [ ]:
import datetime
remaining = cert.not_valid_after_utc - datetime.datetime.now(datetime.UTC)
print('Reste :', remaining.days, 'jours')

**Bonnes pratiques** :

- monitoring expiration : alerte a J-30
- rotation automatique via Let's Encrypt / cert-manager
- ne jamais renouveler a la main en production

---

## 13. mTLS (intro)

**mTLS** (mutual TLS) : le client presente aussi un certificat. Utile pour :

- API service-a-service
- kubernetes (Istio, Linkerd)
- VPN

Le serveur refuse la connexion si le certif client n'est pas signe par une CA de confiance.

---

## Synthese

| Operation | cryptography |
|---|---|
| Cle privee | `rsa.generate_private_key(...)` |
| Self-signed | `x509.CertificateBuilder()` ... `.sign(key, hashes.SHA256())` |
| CSR | `x509.CertificateSigningRequestBuilder()` |
| Signer avec CA | meme builder, `issuer=ca.subject`, `.sign(ca_key, ...)` |
| Inspection | `cert.subject`, `cert.issuer`, `cert.extensions` |
| PEM / DER | `cert.public_bytes(Encoding.PEM/DER)` |

### Regles a retenir

- **TLS 1.2 minimum, 1.3 si possible**.
- **Utiliser Let's Encrypt** en production (sauf mTLS interne).
- **Signer avec SHA-256 ou +**, jamais SHA-1/MD5.
- **Rotation** automatisee.
- **Ne jamais committer une cle privee** dans git.

---

## Exercices

### Exercice 1 (facile) -- Generer une cle RSA

Generez une cle privee RSA 2048 et affichez son PEM.

In [ ]:
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.hazmat.primitives import serialization

# TODO

<details>
<summary>Voir la correction</summary>

```python
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.hazmat.primitives import serialization

k = rsa.generate_private_key(public_exponent=65537, key_size=2048)
pem = k.private_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption(),
)
print(pem.decode()[:120])
```

</details>

### Exercice 2 (facile) -- Certificat auto-signe

Creez un certificat auto-signe pour CN=`test.local`, valide 30 jours.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from cryptography import x509
from cryptography.x509.oid import NameOID
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import rsa
import datetime

k = rsa.generate_private_key(public_exponent=65537, key_size=2048)
subject = issuer = x509.Name([x509.NameAttribute(NameOID.COMMON_NAME, 'test.local')])
now = datetime.datetime.now(datetime.UTC)
cert = (
    x509.CertificateBuilder()
    .subject_name(subject).issuer_name(issuer)
    .public_key(k.public_key())
    .serial_number(x509.random_serial_number())
    .not_valid_before(now).not_valid_after(now + datetime.timedelta(days=30))
    .sign(k, hashes.SHA256())
)
print(cert.not_valid_after_utc)
```

</details>

### Exercice 3 (moyen) -- Fonction generate_certificate

Ecrivez une fonction typee `generate_certificate(common_name: str, days: int) -> tuple[bytes, bytes]` qui retourne `(cert_pem, key_pem)`.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from cryptography import x509
from cryptography.x509.oid import NameOID
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import rsa
import datetime

def generate_certificate(common_name: str, days: int) -> tuple[bytes, bytes]:
    k = rsa.generate_private_key(public_exponent=65537, key_size=2048)
    name = x509.Name([x509.NameAttribute(NameOID.COMMON_NAME, common_name)])
    now = datetime.datetime.now(datetime.UTC)
    cert = (
        x509.CertificateBuilder()
        .subject_name(name).issuer_name(name)
        .public_key(k.public_key())
        .serial_number(x509.random_serial_number())
        .not_valid_before(now).not_valid_after(now + datetime.timedelta(days=days))
        .sign(k, hashes.SHA256())
    )
    key_pem = k.private_bytes(
        serialization.Encoding.PEM,
        serialization.PrivateFormat.PKCS8,
        serialization.NoEncryption(),
    )
    return cert.public_bytes(serialization.Encoding.PEM), key_pem

cert_pem, key_pem = generate_certificate('example.com', 90)
print(cert_pem[:50])
```

</details>

### Exercice 4 (moyen) -- Inspecter un certificat

Ecrivez une fonction `cert_info(pem: bytes) -> dict` qui extrait `subject`, `issuer`, `not_before`, `not_after` et `serial`.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from cryptography import x509

def cert_info(pem: bytes) -> dict:
    cert = x509.load_pem_x509_certificate(pem)
    return {
        'subject': cert.subject.rfc4514_string(),
        'issuer': cert.issuer.rfc4514_string(),
        'not_before': cert.not_valid_before_utc.isoformat(),
        'not_after': cert.not_valid_after_utc.isoformat(),
        'serial': cert.serial_number,
    }
```

</details>

### Exercice 5 (difficile) -- Mini CA

Construisez une mini CA :
1. generez une CA ;
2. creez un CSR pour `api.example.com` ;
3. signez le CSR avec la CA ;
4. verifiez que la signature est valide.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from cryptography import x509
from cryptography.x509.oid import NameOID
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import rsa, padding
import datetime

now = datetime.datetime.now(datetime.UTC)

ca_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
ca_name = x509.Name([x509.NameAttribute(NameOID.COMMON_NAME, 'MiniCA')])
ca_cert = (
    x509.CertificateBuilder()
    .subject_name(ca_name).issuer_name(ca_name)
    .public_key(ca_key.public_key())
    .serial_number(x509.random_serial_number())
    .not_valid_before(now).not_valid_after(now + datetime.timedelta(days=3650))
    .add_extension(x509.BasicConstraints(ca=True, path_length=None), critical=True)
    .sign(ca_key, hashes.SHA256())
)

srv_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
csr = (
    x509.CertificateSigningRequestBuilder()
    .subject_name(x509.Name([x509.NameAttribute(NameOID.COMMON_NAME, 'api.example.com')]))
    .sign(srv_key, hashes.SHA256())
)

signed = (
    x509.CertificateBuilder()
    .subject_name(csr.subject)
    .issuer_name(ca_cert.subject)
    .public_key(csr.public_key())
    .serial_number(x509.random_serial_number())
    .not_valid_before(now).not_valid_after(now + datetime.timedelta(days=365))
    .sign(ca_key, hashes.SHA256())
)

# Verifier la signature
ca_cert.public_key().verify(
    signed.signature,
    signed.tbs_certificate_bytes,
    padding.PKCS1v15(),
    signed.signature_hash_algorithm,
)
print('Signature valide')
```

</details>

---

## Ressources externes

- [cryptography.io](https://cryptography.io/)
- [Let's Encrypt](https://letsencrypt.org/)
- [RFC 8446 - TLS 1.3](https://datatracker.ietf.org/doc/html/rfc8446)
- [Mozilla SSL Config Generator](https://ssl-config.mozilla.org/)